## feature engineering
- turn raw data into model ready inputs
- encode text, scale numbers, handle imbalance

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
import joblib

df = pd.read_csv('../data/raw/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(subset=['TotalCharges'], inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

## drop cols we don't need
- customerID is just unique label, zero predictive signal
- tenure_group only for EDA charts, raw tenure number is more useful

In [4]:
df = df.drop(columns=['customerID'])

print(f"columns remaining: {df.shape[1]}")
print(df.columns.tolist())

columns remaining: 20
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


## split features from target + train/test split
- separate target (churn) from everything else
- hold 20% as a test set the model never sees during training
- stratify keeps same ~26% churn ratio in both splits

In [5]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"training rows: {X_train.shape[0]}")
print(f"test rows:     {X_test.shape[0]}")
print(f"churn rate in train: {y_train.mean():.2%}")
print(f"churn rate in test:  {y_test.mean():.2%}")

training rows: 5625
test rows:     1407
churn rate in train: 26.58%
churn rate in test:  26.58%


## sort cols by type
- diff col types need diff preprocessing so handle them separately
- binary = yes/no cols, cat = multioption cols, num = number cols

In [6]:
binary_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
               'PhoneService', 'PaperlessBilling']

cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport',
            'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'charges_per_month']

print(f"binary cols:     {len(binary_cols)}")
print(f"categorical cols:{len(cat_cols)}")
print(f"numeric cols:    {len(num_cols)}")

binary cols:     6
categorical cols:10
numeric cols:    4


## add engineered feature + encode binary cols
- charges_per_month = total charges / tenure (value per month)
- encode binary yes/no cols to 1/0 before pipeline runs

In [7]:
def add_features(df):
    df = df.copy()
    df['charges_per_month'] = df['TotalCharges'] / (df['tenure'] + 1)
    return df

def encode_binary(df):
    df = df.copy()
    for col in binary_cols:
        if df[col].dtype == object:
            df[col] = (df[col] == 'Yes').astype(int)
    return df

X_train = add_features(encode_binary(X_train))
X_test  = add_features(encode_binary(X_test))

## build preprocessing pipeline
- scaler puts all numeric cols on the same scale so bigger numbers don't dominate
- one hot encoder turns text categories into 0/1 columns model can read
- fit only on training data 

In [8]:
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols),
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f"processed feature shape: {X_train_processed.shape}")

processed feature shape: (5625, 35)


## fix class imbalance w SMOTE
- model trained on 73/26 split will predict "stayed" for everyone and look accurate
- SMOTE creates synthetic churn examples 
- only applied to training data 

In [9]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_processed, y_train)

print(f"before SMOTE — churn rate: {y_train.mean():.2%}")
print(f"after SMOTE  — churn rate: {y_train_resampled.mean():.2%}")
print(f"training rows after SMOTE: {X_train_resampled.shape[0]}")

before SMOTE — churn rate: 26.58%
after SMOTE  — churn rate: 50.00%
training rows after SMOTE: 8260


## save preprocessor to disk
- saves all fitted scaling + encoding params 

In [10]:
import os
os.makedirs('../models', exist_ok=True)

joblib.dump(preprocessor, '../models/preprocessor.pkl')
print("saved to ../models/preprocessor.pkl")

saved to ../models/preprocessor.pkl


## save processed data to disk
- lets future code run standaloen w/o rerunning code 

In [11]:
import numpy as np

os.makedirs('../data/processed', exist_ok=True)

np.save('../data/processed/X_train_resampled.npy', X_train_resampled)
np.save('../data/processed/X_test_processed.npy', X_test_processed)
np.save('../data/processed/y_train_resampled.npy', y_train_resampled)
np.save('../data/processed/y_test.npy', y_test)

print("all processed data saved to data/processed/")

all processed data saved to data/processed/
